# 主成分分析(PCA)(演習)

解説資料は `research-handbook/machine-learning/dimensionality-reduction.md`。
このnotebookは**直接編集せず**、自分の卒研repoにコピーして使うこと(手順は `technical-handbook/colab/use-github-repo.md`)。

「---- ここから課題 ----」の区間を埋めながら上から順に実行する。
解答付き版は `research-handbook/notebooks/pca_solution.ipynb` にあるが、まず自力で取り組むこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 相関のある3次元データ(実質2次元の構造)を作る
rng = np.random.default_rng(0)
latent = rng.normal(0, [3.0, 1.0], (300, 2))              # 2次元の潜在構造
A = np.array([[1.0, 0.2], [0.5, 1.0], [0.3, -0.8]])       # 3次元へ埋め込み
X = latent @ A.T + rng.normal(0, 0.3, (300, 3))           # 観測ノイズ
print("X shape:", X.shape)

## 課題1: PCAの実装

手順は3つ:(1) 中心化、(2) 共分散行列 $S = \frac{1}{N} X_c^\top X_c$、(3) 固有分解して固有値の大きい順に並べる。対称行列には `np.linalg.eigh` を使います(`eig` より速く安定。ただし**昇順**で返るので並べ替えが必要)。

In [ ]:
def pca_fit(X):
    """課題1: 中心化 → 共分散行列 → 固有分解(固有値の大きい順)して
    (固有値, 固有ベクトル行列, 平均) を返す"""
    # ---- ここから課題1 ----
    # (1) 中心化(平均を引く)
    # (2) 共分散行列 S = Xc^T Xc / N
    # (3) np.linalg.eigh で固有分解(昇順で返るので降順に並べ替える)


    # ---- ここまで ----
    return eigvals, eigvecs, X.mean(axis=0)

eigvals, W, mean = pca_fit(X)
print("固有値(分散):", eigvals)
ratio = eigvals / eigvals.sum()
print("寄与率:", ratio, " 累積寄与率:", np.cumsum(ratio))

## 検算: SVDとの一致

PCAはSVDでも計算できます($X_c = U S V^\top$ の $V$ が主成分、$s_i^2/N$ が固有値)。両者が一致することを確認します。固有ベクトルには符号の不定性がある点に注意。

In [ ]:
# 検算1: SVDと一致するか(数値誤差の範囲で)
Xc = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
print("固有値 vs s^2/N の最大誤差:", np.abs(eigvals - s**2 / len(X)).max())
# 固有ベクトルは符号の不定性があるので絶対値で比較
print("固有ベクトル(絶対値)の最大誤差:", np.abs(np.abs(W) - np.abs(Vt.T)).max())

## 課題2・3: 射影と再構成

上位 $M$ 主成分への射影と逆変換を実装し、再構成誤差が**使わなかった固有値の和**と一致すること(`dimensionality-reduction.md` の導出)を数値で確認します。

In [ ]:
def pca_transform(X, W, mean, M):
    """課題2: 上位M主成分への射影 Z = (X - x̄) W_M"""
    # ---- ここから課題2 ----
    return None
    # ---- ここまで ----

def pca_inverse(Z, W, mean):
    """課題3: 逆変換(再構成) X̂ = Z W_M^T + x̄"""
    M = Z.shape[1]
    # ---- ここから課題3 ----
    # 再構成誤差が「捨てた固有値の和」と一致すれば正解
    return None
    # ---- ここまで ----

for M in [1, 2, 3]:
    Z = pca_transform(X, W, mean, M)
    X_rec = pca_inverse(Z, W, mean)
    err = np.mean(((X - X_rec) ** 2).sum(axis=1))
    print(f"M={M}: 再構成誤差 = {err:.4f}(理論値 = 残り固有値の和 = {eigvals[M:].sum():.4f})")

Z2 = pca_transform(X, W, mean, 2)
plt.figure(figsize=(4, 4))
plt.scatter(Z2[:, 0], Z2[:, 1], s=10)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.axis("equal"); plt.title("2D projection")
plt.show()

## 前処理の注意: スケールの混在

単位の違う特徴(mmとm、円と%)を混ぜると、分散の大きい列が第1主成分を支配してしまいます。

In [ ]:
# 前処理の注意: スケールが違う特徴を混ぜるとどうなるか
X_bad = X.copy()
X_bad[:, 2] *= 100                       # 3列目だけ単位が違う(例: mm と m)
ev_bad, W_bad, _ = pca_fit(X_bad)
print("スケール混在時の寄与率:", ev_bad / ev_bad.sum())
# → 第1主成分がスケールの大きい列に支配される。標準化(各列を std=1)してから行うこと
X_std = (X_bad - X_bad.mean(axis=0)) / X_bad.std(axis=0)
ev_std, _, _ = pca_fit(X_std)
print("標準化後の寄与率:", ev_std / ev_std.sum())

## 発展課題

1. 累積寄与率90%に必要な主成分数を返す関数を書き、ノイズの大きさを変えて挙動を見る
2. 自分のRL実験のデータ(例: 方策の隠れ状態、状態訪問データ)を2次元に射影して可視化してみる(`dimensionality-reduction.md` §RL接続)
3. 再構成誤差最小化からPCAを導出する(分散最大化との等価性。`dimensionality-reduction.md` の導出を追う)

## まとめ

- PCA = 中心化 + 共分散行列の固有分解(またはSVD)。寄与率で「何次元残すか」を判断する
- 再構成誤差 = 捨てた固有値の和。この関係が実装の検算になる
- 中心化は必須、スケールが違う特徴は標準化してから
- 確率モデルとしての拡張が確率的PCA、非線形化がカーネルPCA・t-SNE/UMAP(`dimensionality-reduction.md`)